# Refusal follow-up pipeline

Run the steps in order. All file paths are defined in the next cell. `scripts/code.py` (RAW label analysis) is run separately.

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

sys.path.insert(0, str(Path("scripts").resolve()))

import data_import
import clean_data
import rq1_post_refusal_behavior as rq1
import matching

load_dotenv()  # reads LUMEN_API_KEY from .env

WILDCHAT_DIR = Path("data/wildchat")
PROCESSED_DIR = Path("data/processed")

JOINED = PROCESSED_DIR / "refusal_conversations.parquet"
FIRST_REFUSALS = PROCESSED_DIR / "first_refusal_conversations.parquet"
FOLLOWUPS = PROCESSED_DIR / "followup_requests.parquet"
ANNOTATION_SAMPLE = PROCESSED_DIR / "followup_annotation_sample.csv"
PAIRS = PROCESSED_DIR / "request_response_pairs.parquet"
JUDGMENTS = PROCESSED_DIR / "fulfillment_judgments.jsonl"
FULFILLMENT = PROCESSED_DIR / "earliest_fulfillment.parquet"
JUDGE_ANNOTATION_SAMPLE = PROCESSED_DIR / "judge_annotation_sample.csv"
TEST_PAIRS = PROCESSED_DIR / "test_pairs.parquet"
TEST_JUDGMENTS = PROCESSED_DIR / "test_judgments.jsonl"

# NCSA Lumen; the API key is read from LUMEN_API_KEY in .env.
JUDGE_MODEL = "qwen3.8-27b"
JUDGE_URL = "https://lumen.ncsa.illinois.edu/v1"

## 1. Import data
Download WildChat (skip if `data/wildchat` already exists), then join it with RAW refusal labels.

In [ ]:
# data_import.download_wildchat(WILDCHAT_DIR)
data_import.join_with_raw(WILDCHAT_DIR, JOINED)

## 2. Clean data

In [ ]:
# Delete an output file to rebuild it (e.g. after re-running the join).
if FIRST_REFUSALS.exists():
    print("Skipping clean_refusals; found", FIRST_REFUSALS)
else:
    clean_data.clean_refusals(JOINED, FIRST_REFUSALS)

if FOLLOWUPS.exists():
    print("Skipping extract_followups; found", FOLLOWUPS)
else:
    clean_data.extract_followups(FIRST_REFUSALS, FOLLOWUPS)

## 3. RQ1: post-refusal behavior

In [ ]:
rq1.continuation_summary(FIRST_REFUSALS)
rq1.sample_followups(FOLLOWUPS, ANNOTATION_SAMPLE)

## 4. RQ1.2: Match refused requests to later fulfilling responses
Pair each first refusal with every later assistant response in the window, label each pair with the LLM judge, then keep the earliest full fulfillment per conversation. The judge runs on NCSA Lumen: put your Lumen API key in `.env` (copy `.env.example`). Judging appends to `JUDGMENTS` and skips pairs already labeled, so it can be stopped and resumed.

In [ ]:
matching.build_pairs(FIRST_REFUSALS, PAIRS)

### Test run (20 conversations)
Judge every pair from 20 random conversations and read the labels before the full run. Results go to separate test files. Delete `TEST_JUDGMENTS` to judge again, e.g. after changing `JUDGE_MODEL`.

In [ ]:
matching.sample_conversations(PAIRS, TEST_PAIRS, n=20)
matching.judge_pairs(TEST_PAIRS, TEST_JUDGMENTS, model=JUDGE_MODEL, base_url=JUDGE_URL)

In [ ]:
test = matching.load_judgments(TEST_PAIRS, TEST_JUDGMENTS)
print(test["label"].value_counts(dropna=False).to_string())
test[["conversation_id", "candidate_offset", "retry_number", "original_request",
      "candidate_response", "label", "explanation", "evidence"]]

### Full run

In [ ]:
matching.judge_pairs(PAIRS, JUDGMENTS, model=JUDGE_MODEL, base_url=JUDGE_URL)

In [ ]:
matching.earliest_fulfillment(FIRST_REFUSALS, PAIRS, JUDGMENTS, FULFILLMENT)

### Judge validation
Sample judged pairs, fill in `human_label` (one of `full fulfillment`, `partial fulfillment`, `no fulfillment`, `uncertain`) without looking at `judge_label`, then compare.

In [ ]:
matching.sample_pairs_for_annotation(PAIRS, JUDGMENTS, JUDGE_ANNOTATION_SAMPLE)

In [ ]:
matching.judge_agreement(JUDGE_ANNOTATION_SAMPLE)